# Add safety guardrails to a chatbot

Screen both sides of an OpenAI support chat. First enforce a local policy: Ridgeline Outfitters does not send customer email addresses to its chat model or display addresses in replies. Then try hosted Protect for moderation, bias, prompt injection and privacy. The local rule is narrow: an email pattern is not a general privacy or safety detector.

## Install

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit==0.3.7", "openai"], check=True);

## Screen a synthetic support conversation locally

The scanner does not send content to a service. Only the clean chat example needs an OpenAI key.

In [ ]:
from fi.evals.guardrails.scanners.regex import RegexScanner

email_scanner = RegexScanner(patterns=["email"])

def allowed(text):
    return bool(text and text.strip()) and not email_scanner.scan(text).has_matches

assert allowed("When can I return a jacket?")
assert not allowed("Please contact customer@example.invalid.")

In [ ]:
import os
from getpass import getpass

def load_secret(name):
    if os.environ.get(name):
        return
    value = getpass(f"{name}: ")
    if not value.strip():
        raise ValueError(f"{name} is required")
    os.environ[name] = value.strip()

for name in ('OPENAI_API_KEY',):
    load_secret(name)

In [ ]:
from openai import OpenAI

client = OpenAI()
withheld = "I can't display that message."

def guarded_chat(question):
    if not allowed(question):
        return withheld
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You work for Ridgeline Outfitters. Unused gear can be returned within 30 days. Answer only from that policy."},
            {"role": "user", "content": question},
        ],
    )
    answer = completion.choices[0].message.content
    return answer if allowed(answer) else withheld

assert guarded_chat("Please email customer@example.invalid.") == withheld
print(guarded_chat("How long do I have to return unused gear?"))

## Initialize hosted Protect

The remaining model examples require a Future AGI workspace with Protect model serving enabled. For these cells, enter your Future AGI credentials if the environment does not already provide them. A service failure is not a content verdict; withhold the response if a check fails or is incomplete.

In [ ]:
from fi.evals import Protect

# Initialize only when you run a hosted example.
def hosted_protector():
    for name in ("FI_API_KEY", "FI_SECRET_KEY"):
        load_secret(name)
    return Protect()

def check(protector, text, *metrics):
    result = protector.protect(
        text, protect_rules=[{"metric": metric} for metric in metrics],
        action=withheld, reason=True,
    )
    return result["status"] == "passed" and not result["uncompleted_rules"]

## Compare moderation and bias checks

`False` means the message did not clear the check, not necessarily that the classifier found harmful content.

In [ ]:
protector = hosted_protector()
for metric, message in [
    ("toxicity", "You're worthless."),
    ("toxicity", "What are your business hours?"),
    ("bias_detection", "Only men should manage the store."),
    ("bias_detection", "Store managers are selected based on experience."),
]:
    print(metric, "allowed:", check(protector, message, metric))

## Apply multiple rules before and after generation

In [ ]:
protector = hosted_protector()

def protected_chat(question):
    if not check(protector, question, "prompt_injection", "data_privacy_compliance"):
        return withheld
    answer = guarded_chat(question)
    if answer == withheld or not check(protector, answer, "toxicity", "bias_detection"):
        return withheld
    return answer

print(protected_chat("What is your return window?"))

## Try Protect Flash

Flash returns one classification rather than a per-rule breakdown.

In [ ]:
flash = hosted_protector().protect("What are your business hours?", action=withheld, use_flash=True)
print("Flash status:", flash["status"])

[Protect SDK reference](https://docs.futureagi.com/docs/sdk/protect)